# 07 — Final Regular PIT vs Strict/LS PIT Modeling Matrices

This notebook constructs the final clean modeling matrices from the post-06D master dataset.

### Frozen 07 decisions
- Remove `Dataset_Vintage`
- Remove `COVID_Window_Overlap`
- Remove Regular/LS Housing and BLS residual-missing diagnostics
- Preserve the validated DTI model-ready treatment and `DTI_Missing`
- Relabel FICO and DTI risk-band categories using explicit numeric cutoffs
- Rename final macro-cluster fields to modeling-facing names:
  - `Regular_State_Macro_Cluster_A/B`
  - `Regular_MSA_Macro_Cluster_A/B`
  - `LS_State_Macro_Cluster_A/B`
  - `LS_MSA_Macro_Cluster_A/B`
- Retain MSA macro-cluster fallback indicators with simplified `_Fallback` names
- Keep other candidate predictors at the matrix stage; later EDA/model-X selection handles redundancy and ablation.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

FINAL_DTI_STRATEGY = "MEDIAN_DERIVED"   # allowed: MEDIAN_DERIVED, SUPPRESSION_ZERO
ALLOWED_DTI_STRATEGIES = {"MEDIAN_DERIVED", "SUPPRESSION_ZERO"}
assert FINAL_DTI_STRATEGY in ALLOWED_DTI_STRATEGIES

print("="*112)
print("07 — FINAL REGULAR PIT vs STRICT/LS PIT MODEL-READY MATRIX CONSTRUCTION")
print("Final DTI strategy:", FINAL_DTI_STRATEGY)
print("="*112)


07 — FINAL REGULAR PIT vs STRICT/LS PIT MODEL-READY MATRIX CONSTRUCTION
Final DTI strategy: MEDIAN_DERIVED


In [2]:
INPUT_NAME = "SFLLD_feature_engineered_train_validation_MSA_fallback_resolved.parquet"

cwd = Path.cwd().resolve()
search_roots = [cwd] + list(cwd.parents)
relative_candidates = [
    Path("data/output/final_modeling_dataset") / INPUT_NAME,
    Path("data/output") / INPUT_NAME,
    Path(INPUT_NAME),
]

candidate_paths = []
for root in search_roots:
    for rel in relative_candidates:
        p = (root / rel).resolve()
        if p.exists():
            candidate_paths.append(p)
candidate_paths = list(dict.fromkeys(candidate_paths))

if not candidate_paths:
    raise FileNotFoundError(
        f"Could not find {INPUT_NAME}. Run inside the SFLLD project tree "
        "or assign INPUT_PATH manually."
    )

INPUT_PATH = candidate_paths[0]
parts = INPUT_PATH.parts
if "data" in parts:
    i = parts.index("data")
    PROJECT_ROOT = Path(*parts[:i])
else:
    PROJECT_ROOT = INPUT_PATH.parent

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / "final_modeling_dataset"
QA_DIR = PROJECT_ROOT / "data" / "output" / "qa" / "07_modeling_matrix"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
QA_DIR.mkdir(parents=True, exist_ok=True)

REGULAR_OUTPUT = OUTPUT_DIR / "SFLLD_modeling_matrix_regular_PIT_train_validation.parquet"
STRICT_OUTPUT  = OUTPUT_DIR / "SFLLD_modeling_matrix_strict_LS_PIT_train_validation.parquet"

print("Input       :", INPUT_PATH)
print("Regular out :", REGULAR_OUTPUT)
print("Strict out  :", STRICT_OUTPUT)
print("QA dir      :", QA_DIR)


Input       : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\final_modeling_dataset\SFLLD_feature_engineered_train_validation_MSA_fallback_resolved.parquet
Regular out : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\final_modeling_dataset\SFLLD_modeling_matrix_regular_PIT_train_validation.parquet
Strict out  : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\final_modeling_dataset\SFLLD_modeling_matrix_strict_LS_PIT_train_validation.parquet
QA dir      : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\qa\07_modeling_matrix


In [3]:
df = pd.read_parquet(INPUT_PATH)

EXPECTED_ROWS = 209_041
EXPECTED_COLS = 111
EXPECTED_TRAIN = 160_003
EXPECTED_VALIDATION = 49_038

for c in ["Loan_ID", "Target_24M", "Split", "Original_DTI", "DTI_Missing"]:
    assert c in df.columns, f"Required column missing: {c}"

split_norm = df["Split"].astype("string").str.strip().str.lower()
train_mask = split_norm.eq("train")
val_mask = split_norm.eq("validation")

identity = pd.DataFrame({
    "Check":["Rows","Columns","Train rows","Validation rows","Loan_ID unique"],
    "Observed":[len(df),df.shape[1],int(train_mask.sum()),int(val_mask.sum()),bool(df["Loan_ID"].is_unique)],
    "Expected":[EXPECTED_ROWS,EXPECTED_COLS,EXPECTED_TRAIN,EXPECTED_VALIDATION,True],
})
identity["PASS"] = identity["Observed"] == identity["Expected"]
display(identity)
assert identity["PASS"].all(), "Post-06D source identity check failed."


,Check,Observed,Expected,PASS
0,Rows,209041,209041,True
1,Columns,111,111,True
2,Train rows,160003,160003,True
3,Validation rows,49038,49038,True
4,Loan_ID unique,True,True,True


In [4]:

# ======================================================================================
# 3. FROZEN 07 SOURCE SCHEMA
# ======================================================================================

# Dataset_Vintage was already removed from the previous 07 schema.
# COVID_Window_Overlap is now also removed from COMMON_COLS.
COMMON_COLS = [
    'Loan_ID',
    'FICO_Score',
    'First_Payment_Date',
    'First_Time_Homebuyer',
    'Maturity_Date',
    'MSA',
    'MI_Percentage',
    'Number_of_Units',
    'Occupancy_Status',
    'Original_CLTV',
    'Original_DTI',
    'Original_UPB',
    'Original_LTV',
    'Original_Interest_Rate',
    'Channel',
    'Property_State',
    'Property_Type',
    'Postal_Code',
    'Loan_Purpose',
    'Original_Loan_Term',
    'Number_of_Borrowers',
    'Seller_Name',
    'Super_Conforming_Flag',
    'HARP_Indicator',
    'Target_24M',
    'Performance_Window_End',
    'Split',
    'DTI_Missing',
    'FICO_Risk_Band',
    'DTI_Risk_Band',
    'LTV_Risk_Component',
    'CLTV_Risk_Component',
    'CLTV_LTV_Gap',
    'DTI_LTV_Interaction',
    'DTI_Rate_Interaction',
    'UPB_per_Borrower',
    'Estimated_Monthly_PI',
    'High_Leverage',
    'HARP_High_Leverage',
    'FICO_DTI_Interaction',
    'FICO_LTV_Interaction',
    'Spatial_Cluster',
    'Spatial_Cluster_Region',
]

# Residual-missing diagnostics are intentionally NOT included.
REGULAR_ONLY_SOURCE_COLS = [
    'PIT_FHFA_Quarter_Date',
    'PIT_BLS_Month_Date',
    'Housing_HPI',
    'Housing_HPI_Growth_12M',
    'BLS_Unemployment_Rate_Final',
    'BLS_Employment_Growth_12M_Final',
    'BLS_Unemployment_Rate_Change_12M_Final',
    'Regular_State_A_Macro_Cluster_ResolvedBLS',
    'Regular_State_B_Macro_Cluster_ResolvedBLS',
    'Regular_MSA_A_Macro_Cluster_Resolved',
    'Regular_MSA_A_Macro_Cluster_Fallback_Applied',
    'Regular_MSA_B_Macro_Cluster_Resolved',
    'Regular_MSA_B_Macro_Cluster_Fallback_Applied',
]

STRICT_ONLY_SOURCE_COLS = [
    'Strict_PIT_FHFA_Quarter_Date',
    'Strict_PIT_BLS_Month_Date',
    'LS_Housing_HPI',
    'LS_Housing_HPI_Growth_12M',
    'LS_BLS_Unemployment_Rate_Final',
    'LS_BLS_Employment_Growth_12M_Final',
    'LS_BLS_Unemployment_Rate_Change_12M_Final',
    'LS_State_A_Macro_Cluster_ResolvedBLS',
    'LS_State_B_Macro_Cluster_ResolvedBLS',
    'LS_MSA_A_Macro_Cluster_Resolved',
    'LS_MSA_A_Macro_Cluster_Fallback_Applied',
    'LS_MSA_B_Macro_Cluster_Resolved',
    'LS_MSA_B_Macro_Cluster_Fallback_Applied',
]

REGULAR_SOURCE_COLS = COMMON_COLS + REGULAR_ONLY_SOURCE_COLS
STRICT_SOURCE_COLS = COMMON_COLS + STRICT_ONLY_SOURCE_COLS

assert len(COMMON_COLS) == 43
assert len(REGULAR_ONLY_SOURCE_COLS) == 13
assert len(STRICT_ONLY_SOURCE_COLS) == 13
assert len(REGULAR_SOURCE_COLS) == 56 == len(set(REGULAR_SOURCE_COLS))
assert len(STRICT_SOURCE_COLS) == 56 == len(set(STRICT_SOURCE_COLS))

missing_regular = [c for c in REGULAR_SOURCE_COLS if c not in df.columns]
missing_strict = [c for c in STRICT_SOURCE_COLS if c not in df.columns]

spec_check = pd.DataFrame({
    "Matrix": ["Regular PIT", "Strict/LS PIT"],
    "Expected_Columns": [len(REGULAR_SOURCE_COLS), len(STRICT_SOURCE_COLS)],
    "Missing_From_Master": [missing_regular, missing_strict],
    "PASS": [not missing_regular, not missing_strict],
})
display(spec_check)
assert spec_check["PASS"].all(), "Frozen 07 specification cannot be built from the 06D master."

MATRIX_LEVEL_DROPS = [
    "Dataset_Vintage",
    "COVID_Window_Overlap",
    "Housing_Regular_ResidualMissing",
    "BLS_Regular_ResidualMissing",
    "LS_Housing_ResidualMissing",
    "LS_BLS_ResidualMissing",
]

for c in MATRIX_LEVEL_DROPS:
    assert c not in REGULAR_SOURCE_COLS
    assert c not in STRICT_SOURCE_COLS

print("PASS — all frozen 07 matrix-level exclusions are absent from the output schema.")


,Matrix,Expected_Columns,Missing_From_Master,PASS
0,Regular PIT,56,[],True
1,Strict/LS PIT,56,[],True


PASS — all frozen 07 matrix-level exclusions are absent from the output schema.


In [5]:
DTI_INTERACTION_COLS = [
    "DTI_LTV_Interaction",
    "DTI_Rate_Interaction",
    "FICO_DTI_Interaction",
]

dti_missing = df["DTI_Missing"].eq(1)
dti_observed = df["DTI_Missing"].eq(0)

assert df["DTI_Missing"].notna().all()
assert set(df["DTI_Missing"].unique()).issubset({0,1})
assert df["Original_DTI"].notna().all(), "Original_DTI should already be imputed upstream."

audit_rows = []
for col in DTI_INTERACTION_COLS:
    nan_mask = df[col].isna()
    nan_without_missing_dti = nan_mask & ~dti_missing
    populated_despite_missing_dti = (~nan_mask) & dti_missing

    audit_rows.append({
        "Interaction": col,
        "NaN_N": int(nan_mask.sum()),
        "DTI_Missing_N": int(dti_missing.sum()),
        "NaN_with_DTI_Missing_0": int(nan_without_missing_dti.sum()),
        "NonNaN_with_DTI_Missing_1": int(populated_despite_missing_dti.sum()),
        "NaN_IFF_DTI_Missing": bool(nan_mask.equals(dti_missing)),
    })

structural_audit = pd.DataFrame(audit_rows)
display(structural_audit)

assert structural_audit["NaN_IFF_DTI_Missing"].all(), (
    "Structural-missingness equivalence failed. Do NOT fill these NaNs silently."
)
print("PASS — all three interaction NaNs are exactly structural DTI missingness.")


,Interaction,NaN_N,DTI_Missing_N,NaN_with_DTI_Missing_0,NonNaN_with_DTI_Missing_1,NaN_IFF_DTI_Missing
0,DTI_LTV_Interaction,17595,17595,0,0,True
1,DTI_Rate_Interaction,17595,17595,0,0,True
2,FICO_DTI_Interaction,17595,17595,0,0,True


PASS — all three interaction NaNs are exactly structural DTI missingness.


In [6]:
# Median is learned from TRAIN rows with originally observed DTI only.
train_observed_dti = df.loc[train_mask & dti_observed, "Original_DTI"].astype(float)
TRAIN_DTI_MEDIAN = float(train_observed_dti.median())

missing_dti_values = df.loc[dti_missing, "Original_DTI"].astype(float)
max_abs_from_train_median = (
    float(np.max(np.abs(missing_dti_values - TRAIN_DTI_MEDIAN)))
    if len(missing_dti_values) else 0.0
)

dti_imputation_audit = pd.DataFrame([{
    "Train_Observed_DTI_N": int(train_observed_dti.shape[0]),
    "Train_DTI_Median": TRAIN_DTI_MEDIAN,
    "Originally_Missing_DTI_N": int(dti_missing.sum()),
    "Missing_Rows_MaxAbs_From_Train_Median": max_abs_from_train_median,
    "All_Missing_Rows_Equal_Train_Median": bool(
        np.allclose(missing_dti_values, TRAIN_DTI_MEDIAN, rtol=0, atol=1e-12)
    ),
}])
display(dti_imputation_audit)

assert dti_imputation_audit.loc[0, "All_Missing_Rows_Equal_Train_Median"], (
    "Original_DTI on DTI_Missing=1 rows is not uniformly equal to the Train-observed median. "
    "Stop and inspect upstream imputation before constructing median-derived interactions."
)
print("PASS — upstream Original_DTI uses the Train-observed median for all originally missing DTI rows.")


,Train_Observed_DTI_N,Train_DTI_Median,Originally_Missing_DTI_N,Missing_Rows_MaxAbs_From_Train_Median,All_Missing_Rows_Equal_Train_Median
0,144002,35.0,17595,0.0,True


PASS — upstream Original_DTI uses the Train-observed median for all originally missing DTI rows.


In [7]:
formula_map = {
    "DTI_LTV_Interaction": df["Original_DTI"] * df["Original_LTV"],
    "DTI_Rate_Interaction": df["Original_DTI"] * df["Original_Interest_Rate"],
    "FICO_DTI_Interaction": df["FICO_Score"] * df["Original_DTI"],
}

formula_rows = []
for col, expected in formula_map.items():
    mask = dti_observed & df[col].notna() & expected.notna()
    actual = df.loc[mask, col].astype(float)
    exp = expected.loc[mask].astype(float)
    diff = np.abs(actual - exp)
    formula_rows.append({
        "Interaction": col,
        "N_Compared": int(mask.sum()),
        "Max_Abs_Error": float(diff.max()) if len(diff) else np.nan,
        "Mean_Abs_Error": float(diff.mean()) if len(diff) else np.nan,
        "Exact_or_Close": bool(np.allclose(actual, exp, rtol=1e-10, atol=1e-12)),
    })

formula_audit = pd.DataFrame(formula_rows)
display(formula_audit)
assert formula_audit["Exact_or_Close"].all(), "Interaction formula audit failed."


,Interaction,N_Compared,Max_Abs_Error,Mean_Abs_Error,Exact_or_Close
0,DTI_LTV_Interaction,191446,0.0,0.0,True
1,DTI_Rate_Interaction,191446,0.0,0.0,True
2,FICO_DTI_Interaction,191446,0.0,0.0,True


In [8]:
# ======================================================================================
# 7. CONSTRUCT BOTH MODEL-READY DTI STRATEGIES
# ======================================================================================

def apply_dti_strategy(source, strategy):

    out = source.copy()

    miss = out["DTI_Missing"].eq(1)

    # ------------------------------------------------------------------
    # Safety check BEFORE mutation
    #
    # Every interaction NaN must correspond exactly to DTI_Missing = 1.
    # ------------------------------------------------------------------

    for col in DTI_INTERACTION_COLS:

        assert (
            out[col].isna() == miss
        ).all(), (
            f"{col}: structural missingness equivalence changed "
            "before model-ready resolution."
        )

    # ------------------------------------------------------------------
    # Strategy A — structural interaction suppression
    # ------------------------------------------------------------------

    if strategy == "SUPPRESSION_ZERO":

        out.loc[
            miss,
            DTI_INTERACTION_COLS
        ] = 0.0

    # ------------------------------------------------------------------
    # Strategy B — median-derived interactions
    #
    # Original_DTI on DTI_Missing=1 rows has already been verified
    # to equal the frozen Train-observed median.
    # ------------------------------------------------------------------

    elif strategy == "MEDIAN_DERIVED":

        out.loc[
            miss,
            "DTI_LTV_Interaction"
        ] = (
            out.loc[miss, "Original_DTI"]
            * out.loc[miss, "Original_LTV"]
        )

        out.loc[
            miss,
            "DTI_Rate_Interaction"
        ] = (
            out.loc[miss, "Original_DTI"]
            * out.loc[miss, "Original_Interest_Rate"]
        )

        out.loc[
            miss,
            "FICO_DTI_Interaction"
        ] = (
            out.loc[miss, "FICO_Score"]
            * out.loc[miss, "Original_DTI"]
        )

    else:

        raise ValueError(
            f"Unknown DTI strategy: {strategy}"
        )

    # ------------------------------------------------------------------
    # Immediate post-transformation QA
    # ------------------------------------------------------------------

    assert (
        out[DTI_INTERACTION_COLS]
        .isna()
        .sum()
        .sum()
        == 0
    ), (
        f"{strategy}: unresolved DTI interaction NaN remains."
    )

    assert out["DTI_Missing"].equals(
        source["DTI_Missing"]
    ), (
        f"{strategy}: DTI_Missing changed unexpectedly."
    )

    return out


# ======================================================================================
# 7A. BUILD BOTH CHALLENGER REPRESENTATIONS
# ======================================================================================

df_zero = apply_dti_strategy(
    df,
    "SUPPRESSION_ZERO"
)

df_median = apply_dti_strategy(
    df,
    "MEDIAN_DERIVED"
)


# ======================================================================================
# 7B. STRATEGY AUDIT
# ======================================================================================

strategy_audit = []

for strategy, x in [
    ("SUPPRESSION_ZERO", df_zero),
    ("MEDIAN_DERIVED", df_median),
]:

    total_interaction_nan = int(
        x[DTI_INTERACTION_COLS]
        .isna()
        .sum()
        .sum()
    )

    dti_missing_preserved = bool(
        x["DTI_Missing"].equals(
            df["DTI_Missing"]
        )
    )

    observed_rows_unchanged = bool(
        x.loc[
            dti_observed,
            DTI_INTERACTION_COLS
        ].equals(
            df.loc[
                dti_observed,
                DTI_INTERACTION_COLS
            ]
        )
    )

    strategy_audit.append({

        "Strategy":
            strategy,

        "Total_Interaction_NaN":
            total_interaction_nan,

        "DTI_Missing_Preserved":
            dti_missing_preserved,

        "Observed_DTI_Interaction_Rows_Unchanged":
            observed_rows_unchanged,
    })


strategy_audit = pd.DataFrame(
    strategy_audit
)

print("\n" + "=" * 112)
print("DTI MODEL-READY STRATEGY AUDIT")
print("=" * 112)

display(strategy_audit)


# ======================================================================================
# 7C. CORRECT STRATEGY QA
#
# IMPORTANT:
# Do NOT use:
#
#     strategy_audit.drop(columns="Strategy").all(axis=None)
#
# because Total_Interaction_NaN = 0 is a SUCCESS,
# but pandas interprets numeric 0 as False.
# ======================================================================================

assert (
    strategy_audit[
        "Total_Interaction_NaN"
    ] == 0
).all(), (
    "At least one DTI strategy still contains "
    "unresolved interaction NaNs."
)


assert (
    strategy_audit[
        "DTI_Missing_Preserved"
    ]
).all(), (
    "DTI_Missing was not preserved "
    "under at least one strategy."
)


assert (
    strategy_audit[
        "Observed_DTI_Interaction_Rows_Unchanged"
    ]
).all(), (
    "At least one strategy modified interaction values "
    "for originally observed-DTI loans."
)


print(
    "\nPASS — both DTI strategies:"
)

print(
    "  ✓ resolve all structural interaction NaNs"
)

print(
    "  ✓ preserve DTI_Missing"
)

print(
    "  ✓ leave observed-DTI interactions unchanged"
)


# ======================================================================================
# 7D. ADDITIONAL STRATEGY-SPECIFIC QA
# ======================================================================================

missing_mask = df["DTI_Missing"].eq(1)


# ------------------------------------------------------------------
# SUPPRESSION_ZERO
# All three interactions MUST equal zero on originally missing-DTI rows.
# ------------------------------------------------------------------

assert np.allclose(
    df_zero.loc[
        missing_mask,
        DTI_INTERACTION_COLS
    ].to_numpy(dtype=float),
    0.0,
), (
    "SUPPRESSION_ZERO does not produce zero interactions "
    "for every originally missing-DTI row."
)


# ------------------------------------------------------------------
# MEDIAN_DERIVED
# Verify exact formulas on originally missing-DTI rows.
# ------------------------------------------------------------------

expected_dti_ltv = (
    df_median.loc[
        missing_mask,
        "Original_DTI"
    ]
    *
    df_median.loc[
        missing_mask,
        "Original_LTV"
    ]
)

expected_dti_rate = (
    df_median.loc[
        missing_mask,
        "Original_DTI"
    ]
    *
    df_median.loc[
        missing_mask,
        "Original_Interest_Rate"
    ]
)

expected_fico_dti = (
    df_median.loc[
        missing_mask,
        "FICO_Score"
    ]
    *
    df_median.loc[
        missing_mask,
        "Original_DTI"
    ]
)


assert np.allclose(
    df_median.loc[
        missing_mask,
        "DTI_LTV_Interaction"
    ].astype(float),
    expected_dti_ltv.astype(float),
    rtol=1e-10,
    atol=1e-12,
)


assert np.allclose(
    df_median.loc[
        missing_mask,
        "DTI_Rate_Interaction"
    ].astype(float),
    expected_dti_rate.astype(float),
    rtol=1e-10,
    atol=1e-12,
)


assert np.allclose(
    df_median.loc[
        missing_mask,
        "FICO_DTI_Interaction"
    ].astype(float),
    expected_fico_dti.astype(float),
    rtol=1e-10,
    atol=1e-12,
)


print(
    "\nPASS — strategy-specific formula checks:"
)

print(
    "  ✓ SUPPRESSION_ZERO -> all three interactions = 0"
)

print(
    "  ✓ MEDIAN_DERIVED -> all three interactions rebuilt "
    "from Train-median-imputed Original_DTI"
)


# ======================================================================================
# 7E. SELECT CANONICAL MODEL-READY REPRESENTATION
# ======================================================================================

if FINAL_DTI_STRATEGY == "MEDIAN_DERIVED":

    df_modelready = df_median

elif FINAL_DTI_STRATEGY == "SUPPRESSION_ZERO":

    df_modelready = df_zero

else:

    raise ValueError(
        f"Unsupported FINAL_DTI_STRATEGY: "
        f"{FINAL_DTI_STRATEGY}"
    )


print("\n" + "-" * 112)

print(
    f"Canonical DTI strategy selected : "
    f"{FINAL_DTI_STRATEGY}"
)

print(
    f"DTI_Missing rows               : "
    f"{int(missing_mask.sum()):,}"
)

print(
    "Remaining DTI interaction NaN  : "
    f"{int(df_modelready[DTI_INTERACTION_COLS].isna().sum().sum()):,}"
)

print("-" * 112)



DTI MODEL-READY STRATEGY AUDIT


,Strategy,Total_Interaction_NaN,DTI_Missing_Preserved,Observed_DTI_Interaction_Rows_Unchanged
0,SUPPRESSION_ZERO,0,True,True
1,MEDIAN_DERIVED,0,True,True



PASS — both DTI strategies:
  ✓ resolve all structural interaction NaNs
  ✓ preserve DTI_Missing
  ✓ leave observed-DTI interactions unchanged

PASS — strategy-specific formula checks:
  ✓ SUPPRESSION_ZERO -> all three interactions = 0
  ✓ MEDIAN_DERIVED -> all three interactions rebuilt from Train-median-imputed Original_DTI

----------------------------------------------------------------------------------------------------------------
Canonical DTI strategy selected : MEDIAN_DERIVED
DTI_Missing rows               : 17,595
Remaining DTI interaction NaN  : 0
----------------------------------------------------------------------------------------------------------------


In [9]:

# ======================================================================================
# 8. CONSTRUCT FINAL MATRICES, RENAME MACRO FIELDS, RELABEL RISK BANDS
# ======================================================================================

regular = df_modelready.loc[:, REGULAR_SOURCE_COLS].copy()
strict  = df_modelready.loc[:, STRICT_SOURCE_COLS].copy()

REGULAR_RENAME = {
    "Regular_State_A_Macro_Cluster_ResolvedBLS": "Regular_State_Macro_Cluster_A",
    "Regular_State_B_Macro_Cluster_ResolvedBLS": "Regular_State_Macro_Cluster_B",
    "Regular_MSA_A_Macro_Cluster_Resolved": "Regular_MSA_Macro_Cluster_A",
    "Regular_MSA_B_Macro_Cluster_Resolved": "Regular_MSA_Macro_Cluster_B",
    "Regular_MSA_A_Macro_Cluster_Fallback_Applied": "Regular_MSA_Macro_Cluster_A_Fallback",
    "Regular_MSA_B_Macro_Cluster_Fallback_Applied": "Regular_MSA_Macro_Cluster_B_Fallback",
}

STRICT_RENAME = {
    "LS_State_A_Macro_Cluster_ResolvedBLS": "LS_State_Macro_Cluster_A",
    "LS_State_B_Macro_Cluster_ResolvedBLS": "LS_State_Macro_Cluster_B",
    "LS_MSA_A_Macro_Cluster_Resolved": "LS_MSA_Macro_Cluster_A",
    "LS_MSA_B_Macro_Cluster_Resolved": "LS_MSA_Macro_Cluster_B",
    "LS_MSA_A_Macro_Cluster_Fallback_Applied": "LS_MSA_Macro_Cluster_A_Fallback",
    "LS_MSA_B_Macro_Cluster_Fallback_Applied": "LS_MSA_Macro_Cluster_B_Fallback",
}

regular.rename(columns=REGULAR_RENAME, inplace=True)
strict.rename(columns=STRICT_RENAME, inplace=True)

# Relabel only — no re-binning and no cutoff recomputation.
FICO_BAND_LABEL_MAP = {
    "Very_High_Risk": "FICO_<660",
    "High_Risk": "FICO_660_699",
    "Moderate_Risk": "FICO_700_739",
    "Low_Risk": "FICO_740_759",
    "Very_Low_Risk": "FICO_760plus",
}

DTI_BAND_LABEL_MAP = {
    "Missing_DTI": "DTI_Missing",
    "Low_DTI": "DTI_<25",
    "Moderate_DTI": "DTI_25_34",
    "Elevated_DTI": "DTI_35_39",
    "High_DTI": "DTI_40plus",
}

def relabel_exact(series, mapping, name):
    before = series.astype("string")
    unknown = sorted(set(before.dropna().unique()) - set(mapping))
    assert not unknown, f"{name}: unexpected pre-07 labels: {unknown}"
    out = before.map(mapping)
    assert out.notna().sum() == before.notna().sum(), f"{name}: relabel introduced missing values."
    return out

for x in (regular, strict):
    x["FICO_Risk_Band"] = relabel_exact(x["FICO_Risk_Band"], FICO_BAND_LABEL_MAP, "FICO_Risk_Band")
    x["DTI_Risk_Band"] = relabel_exact(x["DTI_Risk_Band"], DTI_BAND_LABEL_MAP, "DTI_Risk_Band")

REGULAR_COLS = [REGULAR_RENAME.get(c, c) for c in REGULAR_SOURCE_COLS]
STRICT_COLS  = [STRICT_RENAME.get(c, c) for c in STRICT_SOURCE_COLS]

assert regular.columns.tolist() == REGULAR_COLS
assert strict.columns.tolist() == STRICT_COLS
assert regular.shape == (EXPECTED_ROWS, len(REGULAR_COLS))
assert strict.shape == (EXPECTED_ROWS, len(STRICT_COLS))

print(f"Regular PIT : {regular.shape[0]:,} rows × {regular.shape[1]} columns")
print(f"Strict/LS   : {strict.shape[0]:,} rows × {strict.shape[1]} columns")


Regular PIT : 209,041 rows × 56 columns
Strict/LS   : 209,041 rows × 56 columns


In [10]:

# ======================================================================================
# 9. MANDATORY PRE-SAVE QA
# ======================================================================================

EXCLUDED_PROVENANCE = [
    "Spatial_Latitude", "Spatial_Longitude",
    "FHFA_State_HPI", "FHFA_State_HPI_Growth_12M", "FHFA_MSA_HPI", "FHFA_MSA_HPI_Growth_12M",
    "LS_FHFA_State_HPI", "LS_FHFA_State_HPI_Growth_12M", "LS_FHFA_MSA_HPI", "LS_FHFA_MSA_HPI_Growth_12M",
    "Housing_Geography_Source", "Housing_Fallback_Level",
    "LS_Housing_Geography_Source", "LS_Housing_Fallback_Level",
    "BLS_State_Fallback_Applied", "BLS_Fallback_Level", "BLS_Geography_Source_Final",
    "LS_BLS_State_Fallback_Applied", "LS_BLS_Fallback_Level", "LS_BLS_Geography_Source_Final",
]

RAW_DIAGNOSTICS = [c for c in df.columns if (
    c.endswith("_Macro_Cluster_Raw")
    or c.endswith("_Macro_Cluster_Resolution_Source")
    or c.endswith("_Macro_Cluster_Fallback_Confidence")
)]

OBSOLETE_MSA = [
    "Regular_MSA_A_Macro_Cluster_ResolvedBLS",
    "Regular_MSA_B_Macro_Cluster_ResolvedBLS",
    "LS_MSA_A_Macro_Cluster_ResolvedBLS",
    "LS_MSA_B_Macro_Cluster_ResolvedBLS",
]

OLD_FINAL_NAMES = list(REGULAR_RENAME) + list(STRICT_RENAME)
FORBIDDEN = set(EXCLUDED_PROVENANCE + RAW_DIAGNOSTICS + OBSOLETE_MSA + OLD_FINAL_NAMES + MATRIX_LEVEL_DROPS)

REG_FINAL = [
    "Regular_State_Macro_Cluster_A",
    "Regular_State_Macro_Cluster_B",
    "Regular_MSA_Macro_Cluster_A",
    "Regular_MSA_Macro_Cluster_B",
]
REG_FB = [
    "Regular_MSA_Macro_Cluster_A_Fallback",
    "Regular_MSA_Macro_Cluster_B_Fallback",
]
LS_FINAL = [
    "LS_State_Macro_Cluster_A",
    "LS_State_Macro_Cluster_B",
    "LS_MSA_Macro_Cluster_A",
    "LS_MSA_Macro_Cluster_B",
]
LS_FB = [
    "LS_MSA_Macro_Cluster_A_Fallback",
    "LS_MSA_Macro_Cluster_B_Fallback",
]

def same(a, b):
    return a.reset_index(drop=True).equals(b.reset_index(drop=True))

checks = []
def chk(name, ok, detail=""):
    checks.append({"Check": name, "PASS": bool(ok), "Detail": detail})

chk("Regular row count unchanged", len(regular) == len(df))
chk("Strict row count unchanged", len(strict) == len(df))

for c in ["Loan_ID", "Target_24M", "Split"]:
    chk(f"Regular {c} unchanged", same(regular[c], df[c]))
    chk(f"Strict {c} unchanged", same(strict[c], df[c]))

chk("Regular exact schema/order", regular.columns.tolist() == REGULAR_COLS)
chk("Strict exact schema/order", strict.columns.tolist() == STRICT_COLS)

chk("Regular final column count", regular.shape[1] == len(REGULAR_COLS), f"Observed={regular.shape[1]}")
chk("Strict final column count", strict.shape[1] == len(STRICT_COLS), f"Observed={strict.shape[1]}")

for c in MATRIX_LEVEL_DROPS:
    chk(f"{c} absent — Regular", c not in regular.columns)
    chk(f"{c} absent — Strict", c not in strict.columns)

chk("No forbidden provenance/diagnostics — Regular", not FORBIDDEN.intersection(regular.columns),
    str(sorted(FORBIDDEN.intersection(regular.columns))))
chk("No forbidden provenance/diagnostics — Strict", not FORBIDDEN.intersection(strict.columns),
    str(sorted(FORBIDDEN.intersection(strict.columns))))

chk("Final Regular macro fields retained", set(REG_FINAL).issubset(regular.columns))
chk("Regular MSA fallback flags retained", set(REG_FB).issubset(regular.columns))
chk("Final Strict macro fields retained", set(LS_FINAL).issubset(strict.columns))
chk("Strict MSA fallback flags retained", set(LS_FB).issubset(strict.columns))

chk("Regular DTI interactions have zero NaN",
    regular[DTI_INTERACTION_COLS].isna().sum().sum() == 0)
chk("Strict DTI interactions have zero NaN",
    strict[DTI_INTERACTION_COLS].isna().sum().sum() == 0)

chk("Regular total NaN = 0", regular.isna().sum().sum() == 0)
chk("Strict total NaN = 0", strict.isna().sum().sum() == 0)

expected_fico_labels = set(FICO_BAND_LABEL_MAP.values())
expected_dti_labels = set(DTI_BAND_LABEL_MAP.values())

for name, x in [("Regular", regular), ("Strict", strict)]:
    chk(f"{name} FICO labels valid",
        set(x["FICO_Risk_Band"].astype("string").dropna().unique()).issubset(expected_fico_labels))
    chk(f"{name} DTI labels valid",
        set(x["DTI_Risk_Band"].astype("string").dropna().unique()).issubset(expected_dti_labels))

    sn = x["Split"].astype("string").str.strip().str.lower()
    chk(f"{name} Train rows", int(sn.eq("train").sum()) == EXPECTED_TRAIN)
    chk(f"{name} Validation rows", int(sn.eq("validation").sum()) == EXPECTED_VALIDATION)

qa = pd.DataFrame(checks)
display(qa)
assert qa["PASS"].all(), "MANDATORY PRE-SAVE QA FAILED."

print("=" * 100)
print("PASS — MANDATORY PRE-SAVE QA")
print("=" * 100)
print(f"Regular shape : {regular.shape}")
print(f"Strict shape  : {strict.shape}")
print(f"Regular NaN   : {int(regular.isna().sum().sum())}")
print(f"Strict NaN    : {int(strict.isna().sum().sum())}")


,Check,PASS,Detail
0,Regular row count unchanged,True,
1,Strict row count unchanged,True,
2,Regular Loan_ID unchanged,True,
3,Strict Loan_ID unchanged,True,
4,Regular Target_24M unchanged,True,
5,Strict Target_24M unchanged,True,
6,Regular Split unchanged,True,
7,Strict Split unchanged,True,
8,Regular exact schema/order,True,
9,Strict exact schema/order,True,


PASS — MANDATORY PRE-SAVE QA
Regular shape : (209041, 56)
Strict shape  : (209041, 56)
Regular NaN   : 0
Strict NaN    : 0


In [11]:
def schema_table(x):
    out=pd.DataFrame({
        "Position":np.arange(1,x.shape[1]+1),
        "Column":x.columns,
        "Dtype":[str(x[c].dtype) for c in x.columns],
        "Missing_N":[int(x[c].isna().sum()) for c in x.columns],
    })
    out["Missing_Pct"]=out["Missing_N"]/len(x)*100
    return out

regular_schema=schema_table(regular)
strict_schema=schema_table(strict)

regular.to_parquet(REGULAR_OUTPUT,index=False)
strict.to_parquet(STRICT_OUTPUT,index=False)

regular_schema.to_csv(QA_DIR/"07_regular_PIT_schema.csv",index=False)
strict_schema.to_csv(QA_DIR/"07_strict_LS_PIT_schema.csv",index=False)
identity.to_csv(QA_DIR/"07_source_identity_QA.csv",index=False)
structural_audit.to_csv(QA_DIR/"07_DTI_structural_missingness_audit.csv",index=False)
dti_imputation_audit.to_csv(QA_DIR/"07_DTI_train_median_imputation_audit.csv",index=False)
formula_audit.to_csv(QA_DIR/"07_DTI_interaction_formula_audit.csv",index=False)
strategy_audit.to_csv(QA_DIR/"07_DTI_strategy_construction_audit.csv",index=False)
qa.to_csv(QA_DIR/"07_pre_save_QA.csv",index=False)

print("Saved:")
print(" -", REGULAR_OUTPUT)
print(" -", STRICT_OUTPUT)


Saved:
 - C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\final_modeling_dataset\SFLLD_modeling_matrix_regular_PIT_train_validation.parquet
 - C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\final_modeling_dataset\SFLLD_modeling_matrix_strict_LS_PIT_train_validation.parquet


In [12]:

# ======================================================================================
# 11. POST-SAVE RELOAD QA
# ======================================================================================

r = pd.read_parquet(REGULAR_OUTPUT)
s = pd.read_parquet(STRICT_OUTPUT)

post = []
def pchk(name, ok, detail=""):
    post.append({"Check": name, "PASS": bool(ok), "Detail": detail})

for name, x, mem, cols in [
    ("Regular", r, regular, REGULAR_COLS),
    ("Strict", s, strict, STRICT_COLS),
]:
    pchk(f"{name} rows", len(x) == EXPECTED_ROWS)
    pchk(f"{name} columns", x.shape[1] == len(cols), f"Observed={x.shape[1]}")
    pchk(f"{name} exact schema/order", x.columns.tolist() == cols)
    pchk(f"{name} total NaN = 0", x.isna().sum().sum() == 0)
    pchk(f"{name} DTI interaction NaN = 0",
         x[DTI_INTERACTION_COLS].isna().sum().sum() == 0)

    for c in MATRIX_LEVEL_DROPS:
        pchk(f"{name} {c} absent", c not in x.columns)

    for c in ["Loan_ID", "Target_24M", "Split", "DTI_Missing", "Original_DTI",
              "FICO_Risk_Band", "DTI_Risk_Band"]:
        pchk(f"{name} exact persisted {c}", same(x[c], mem[c]))

    for c in DTI_INTERACTION_COLS:
        pchk(f"{name} exact persisted {c}", same(x[c], mem[c]))

for c in REG_FINAL + REG_FB:
    pchk(f"Regular exact persisted {c}", c in r.columns and same(r[c], regular[c]))

for c in LS_FINAL + LS_FB:
    pchk(f"Strict exact persisted {c}", c in s.columns and same(s[c], strict[c]))

pchk("Regular forbidden fields absent", not FORBIDDEN.intersection(r.columns),
     str(sorted(FORBIDDEN.intersection(r.columns))))
pchk("Strict forbidden fields absent", not FORBIDDEN.intersection(s.columns),
     str(sorted(FORBIDDEN.intersection(s.columns))))

post_qa = pd.DataFrame(post)
display(post_qa)
post_qa.to_csv(QA_DIR / "07_post_save_QA.csv", index=False)

assert post_qa["PASS"].all(), "POST-SAVE QA FAILED."
print("PASS — post-save reload QA.")


,Check,PASS,Detail
0,Regular rows,True,
1,Regular columns,True,Observed=56
2,Regular exact schema/order,True,
3,Regular total NaN = 0,True,
4,Regular DTI interaction NaN = 0,True,
5,Regular Dataset_Vintage absent,True,
6,Regular COVID_Window_Overlap absent,True,
7,Regular Housing_Regular_ResidualMissing absent,True,
8,Regular BLS_Regular_ResidualMissing absent,True,
9,Regular LS_Housing_ResidualMissing absent,True,


PASS — post-save reload QA.


In [13]:

# ======================================================================================
# 12. FINAL SUMMARY
# ======================================================================================

summary = pd.DataFrame([{
    "Source": INPUT_PATH.name,
    "Source_Rows": len(df),
    "Source_Columns": df.shape[1],
    "Train_Rows": int(train_mask.sum()),
    "Validation_Rows": int(val_mask.sum()),
    "DTI_Strategy": FINAL_DTI_STRATEGY,
    "Train_DTI_Median": TRAIN_DTI_MEDIAN,
    "DTI_Missing_N": int(dti_missing.sum()),
    "Regular_Output": REGULAR_OUTPUT.name,
    "Regular_Columns": r.shape[1],
    "Regular_Total_NaN": int(r.isna().sum().sum()),
    "Strict_Output": STRICT_OUTPUT.name,
    "Strict_Columns": s.shape[1],
    "Strict_Total_NaN": int(s.isna().sum().sum()),
    "Mandatory_QA_Passed": bool(qa.PASS.all()),
    "Post_Save_QA_Passed": bool(post_qa.PASS.all()),
}])

display(summary)
summary.to_csv(QA_DIR / "07_final_summary.csv", index=False)

assert df.shape == (EXPECTED_ROWS, EXPECTED_COLS)
assert r.shape == (EXPECTED_ROWS, len(REGULAR_COLS))
assert s.shape == (EXPECTED_ROWS, len(STRICT_COLS))
assert r.isna().sum().sum() == 0
assert s.isna().sum().sum() == 0
assert qa.PASS.all() and post_qa.PASS.all()

print("=" * 112)
print("07 COMPLETE — FINAL REGULAR PIT AND STRICT/LS PIT MODEL-READY MATRICES VERIFIED")
print("DTI strategy :", FINAL_DTI_STRATEGY)
print(f"Rows         : {EXPECTED_ROWS:,} each")
print(f"Columns      : {len(REGULAR_COLS)} Regular / {len(STRICT_COLS)} Strict")
print("Total NaN    : 0 each")
print("STATUS       : PASS")
print("=" * 112)


,Source,Source_Rows,Source_Columns,Train_Rows,Validation_Rows,DTI_Strategy,Train_DTI_Median,DTI_Missing_N,Regular_Output,Regular_Columns,Regular_Total_NaN,Strict_Output,Strict_Columns,Strict_Total_NaN,Mandatory_QA_Passed,Post_Save_QA_Passed
0,SFLLD_feature_engineered_train_validation_MSA_...,209041,111,160003,49038,MEDIAN_DERIVED,35.0,17595,SFLLD_modeling_matrix_regular_PIT_train_valida...,56,0,SFLLD_modeling_matrix_strict_LS_PIT_train_vali...,56,0,True,True


07 COMPLETE — FINAL REGULAR PIT AND STRICT/LS PIT MODEL-READY MATRICES VERIFIED
DTI strategy : MEDIAN_DERIVED
Rows         : 209,041 each
Columns      : 56 Regular / 56 Strict
Total NaN    : 0 each
STATUS       : PASS
